In [1]:
import numpy as np

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [2]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        self.output = input
        return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [3]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        """
        Adds a module to the container.
        """
        self.modules.append(module)

    def updateOutput(self, input):
        """
        Basic workflow of FORWARD PASS:

            y_0    = module[0].forward(input)
            y_1    = module[1].forward(y_0)
            ...
            output = module[n-1].forward(y_{n-2})


        Just write a little loop.
        """

        # Your code goes here. 
        current_output = input
        for module in self.modules:
            current_output = module.updateOutput(current_output)
            
        self.output = current_output
        return self.output
        
    def updateGradInput(self, input, gradOutput):
        current_grad = gradOutput

        for i in range(len(self.modules) - 1, -1, -1):
            if i == 0:
                current_input = input
            else:
                current_input = self.modules[i - 1].output
            current_grad = self.modules[i].updateGradInput(current_input, current_grad)

        self.gradInput = current_grad
        return self.gradInput

    def backward(self, input, gradOutput):
        """
        Workflow of BACKWARD PASS:

            g_{n-1} = module[n-1].backward(y_{n-2}, gradOutput)
            g_{n-2} = module[n-2].backward(y_{n-3}, g_{n-1})
            ...
            g_1 = module[1].backward(y_0, g_2)
            gradInput = module[0].backward(input, g_1)


        !!!

        To ech module you need to provide the input, module saw while forward pass,
        it is used while computing gradients.
        Make sure that the input for `i-th` layer the output of `module[i]` (just the same input as in forward pass)
        and NOT `input` to this Sequential module.

        !!!

        """
        # Your code goes here. 
        current_grad = gradOutput

        for i in range(len(self.modules) - 1, -1, -1):
            if i == 0:
                current_input = input
            else:
                current_input = self.modules[i - 1].output

            self.modules[i].updateGradInput(current_input, current_grad)
            self.modules[i].accGradParameters(current_input, current_grad)
            current_grad = self.modules[i].gradInput
            
        self.gradInput = current_grad
        
        return self.gradInput


    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        """
        Should gather all parameters in a list.
        """
        params = []
        for module in self.modules:
            params.append(module.getParameters())
        return params

    def getGradParameters(self):
        """
        Should gather all gradients w.r.t parameters in a list.
        """
        params = []
        for module in self.modules:
            params.append(module.getGradParameters())
        return params

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self,x):
        return self.modules.__getitem__(x)

    def train(self):
        """
        Propagates training parameter through all modules
        """
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        """
        Propagates training parameter through all modules
        """
        self.training = False
        for module in self.modules:
            module.evaluate()

# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [4]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = input @ self.W + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput @ self.W.T
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradW += input.T @ gradOutput
        self.gradb += gradOutput.sum(axis=0)
        pass

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q

In [5]:
model = Sequential()
model.add(Linear(2, 2))

x = np.array([[1.0, 2.0]])
out = model.forward(x)
print(out)

[[1.31600873 0.32031266]]


## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [6]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        # Your code goes here. ################################################
        self.output = np.exp(self.output)
        self.output = self.output / self.output.sum(axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = self.output * (
            gradOutput - np.sum(gradOutput * self.output, axis=1, keepdims=True)
        )
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

In [7]:
sm = SoftMax()
x = np.array([[1.0, 2.0, 3.0]])
out = sm.forward(x)
print(out)
print(out.sum(axis=1))

[[0.09003057 0.24472847 0.66524096]]
[1.]


## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [8]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        # Your code goes here. ################################################
        exp_sum = np.sum(np.exp(self.output), axis=1, keepdims=True)
        self.output = self.output - np.log(exp_sum)
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput - np.exp(self.output) * np.sum(gradOutput, axis=1, keepdims=True)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

In [9]:
lsm = LogSoftMax()
x = np.array([[1.0, 2.0, 3.0]])
out = lsm.forward(x)

print(out)
print(np.exp(out).sum(axis=1))

[[-2.40760596 -1.40760596 -0.40760596]]
[1.]


## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [10]:
class BatchNormalization(Module):
    EPS = 1e-3

    def __init__(self, alpha=0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        # use self.EPS please
        if self.moving_mean is None:
            self.moving_mean = np.zeros(input.shape[1], dtype=input.dtype)
        if self.moving_variance is None:
            self.moving_variance = np.ones(input.shape[1], dtype=input.dtype)

        if self.training:
            self.batch_mean = np.mean(input, axis=0)
            self.batch_variance = np.var(input, axis=0)

            self.centered_input = input - self.batch_mean
            self.std = np.sqrt(self.batch_variance + self.EPS)
            self.normalized_input = self.centered_input / self.std

            self.moving_mean = self.alpha * self.moving_mean + (1 - self.alpha) * self.batch_mean
            self.moving_variance = self.alpha * self.moving_variance + (1 - self.alpha) * self.batch_variance

            self.output = self.normalized_input
        else:
            self.output = (input - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)

        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if self.training:
            batch_size = input.shape[0]
            x_hat = self.normalized_input
            inv_std = 1.0 / self.std

            self.gradInput = (1.0 / batch_size) * inv_std * (
                batch_size * gradOutput
                - np.sum(gradOutput, axis=0)
                - x_hat * np.sum(gradOutput * x_hat, axis=0)
            )
        else:
            self.gradInput = gradOutput / np.sqrt(self.moving_variance + self.EPS)

        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [11]:
bn = BatchNormalization(alpha=0.9)
x = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])

out = bn.forward(x)
print(out)
print(out.mean(axis=0))
print(out.var(axis=0))

[[-1.2245153 -1.2245153]
 [ 0.         0.       ]
 [ 1.2245153  1.2245153]]
[0. 0.]
[0.99962514 0.99962514]


In [12]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

<>:2: SyntaxWarning: invalid escape sequence '\g'
<>:2: SyntaxWarning: invalid escape sequence '\g'
/var/folders/4p/w6fy12j92d56479m0fv7_tww0000gn/T/ipykernel_56570/3092293347.py:2: SyntaxWarning: invalid escape sequence '\g'
  """


Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [13]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if self.training:
            self.mask = (np.random.rand(*input.shape) > self.p).astype(input.dtype)
            self.output = input * self.mask / (1 - self.p)
        else:
            self.mask = None
            self.output = input
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if self.training:
            self.gradInput = gradOutput * self.mask / (1 - self.p)
        else:
            self.gradInput = gradOutput
            
        return self.gradInput

    def __repr__(self):
        return "Dropout"

In [14]:
do = Dropout(p=0.5)
x = np.ones((4, 5))

out = do.forward(x)
print(out)

[[0. 2. 0. 0. 2.]
 [0. 2. 2. 0. 2.]
 [2. 0. 0. 2. 0.]
 [2. 0. 2. 0. 0.]]


#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [15]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels

        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = kernel_size

        if isinstance(stride, int):
            self.stride = (stride, stride)
        else:
            self.stride = stride

        self.padding = padding
        self.padding_mode = padding_mode
        self.use_bias = bias

        kh, kw = self.kernel_size
        limit = 1. / np.sqrt(in_channels * kh * kw)

        self.weight = np.random.uniform(
            -limit, limit, size=(out_channels, in_channels, kh, kw)
        )
        self.gradWeight = np.zeros_like(self.weight)

        if bias:
            self.bias = np.random.uniform(-limit, limit, size=(out_channels,))
            self.gradBias = np.zeros_like(self.bias)
        else:
            self.bias = None
            self.gradBias = None

    def _get_padding(self, h, w):
        if self.padding == 'valid':
            return (0, 0, 0, 0)

        if self.padding == 'same':
            sh, sw = self.stride
            kh, kw = self.kernel_size

            out_h = int(np.ceil(h / sh))
            out_w = int(np.ceil(w / sw))

            pad_h_total = max((out_h - 1) * sh + kh - h, 0)
            pad_w_total = max((out_w - 1) * sw + kw - w, 0)

            pad_top = pad_h_total // 2
            pad_bottom = pad_h_total - pad_top
            pad_left = pad_w_total // 2
            pad_right = pad_w_total - pad_left

            return (pad_top, pad_bottom, pad_left, pad_right)

        if isinstance(self.padding, int):
            return (self.padding, self.padding, self.padding, self.padding)

        if isinstance(self.padding, tuple):
            if len(self.padding) == 2:
                ph, pw = self.padding
                return (ph, ph, pw, pw)
            elif len(self.padding) == 4:
                return self.padding

        raise ValueError(f"Unsupported padding: {self.padding}")

    def _pad_input(self, input):
        batch_size, in_channels, h, w = input.shape
        pad_top, pad_bottom, pad_left, pad_right = self._get_padding(h, w)

        if self.padding_mode == 'zeros':
            padded = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='constant'
            )
        elif self.padding_mode == 'reflect':
            padded = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='reflect'
            )
        elif self.padding_mode == 'replicate':
            padded = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='edge'
            )
        else:
            raise ValueError(f"Unsupported padding_mode: {self.padding_mode}")

        return padded, (pad_top, pad_bottom, pad_left, pad_right)

    def updateOutput(self, input):
        self.input = input
        x, self.pad_info = self._pad_input(input)
        self.padded_input = x
        
        batch_size = x.shape[0]
        h_in, w_in = x.shape[2], x.shape[3]
        
        kh, kw = self.kernel_size
        sh, sw = self.stride
        
        if self.padding == 'same':
            out_h = int(np.ceil(input.shape[2] / sh))
            out_w = int(np.ceil(input.shape[3] / sw))
        else:
            out_h = int(np.floor((h_in - kh) / sh)) + 1
            out_w = int(np.floor((w_in - kw) / sw)) + 1
            
        self.output = np.zeros((batch_size, self.out_channels, out_h, out_w))
        
        for n in range(batch_size):
            for oc in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):
                        hs = i * sh
                        ws = j * sw
                        window = x[n, :, hs:hs + kh, ws:ws + kw]
                        self.output[n, oc, i, j] = np.sum(window * self.weight[oc])
                        if self.use_bias:
                            self.output[n, oc, i, j] += self.bias[oc]
        
        return self.output

    def updateGradInput(self, input, gradOutput):
        x = self.padded_input
        batch_size, _, h_in, w_in = x.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        out_h, out_w = gradOutput.shape[2], gradOutput.shape[3]
        
        gradInput_padded = np.zeros_like(x)
        
        for n in range(batch_size):
            for oc in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):
                        hs = i * sh
                        ws = j * sw
                        for ic in range(self.in_channels):
                            gradInput_padded[n, ic, hs:hs + kh, ws:ws + kw] += (
                                gradOutput[n, oc, i, j] * self.weight[oc, ic]
                            )
        pad_top, pad_bottom, pad_left, pad_right = self.pad_info
        
        h_start = pad_top
        h_end = h_in - pad_bottom if pad_bottom > 0 else h_in
        w_start = pad_left
        w_end = w_in - pad_right if pad_right > 0 else w_in
        
        self.gradInput = gradInput_padded[:, :, h_start:h_end, w_start:w_end]
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        x = self.padded_input
        kh, kw = self.kernel_size
        sh, sw = self.stride
        batch_size = x.shape[0]
        out_h, out_w = gradOutput.shape[2], gradOutput.shape[3]

        for n in range(batch_size):
            for oc in range(self.out_channels):
                for i in range(out_h):
                    for j in range(out_w):
                        hs = i * sh
                        ws = j * sw
                        window = x[n, :, hs:hs + kh, ws:ws + kw]
                        self.gradWeight[oc] += gradOutput[n, oc, i, j] * window
                        if self.use_bias:
                            self.gradBias[oc] += gradOutput[n, oc, i, j]

    def zeroGradParameters(self):
        self.gradWeight.fill(0)
        if self.use_bias:
            self.gradBias.fill(0)

    def getParameters(self):
        if self.use_bias:
            return [self.weight, self.bias]
        return [self.weight]

    def getGradParameters(self):
        if self.use_bias:
            return [self.gradWeight, self.gradBias]
        return [self.gradWeight]

    def __repr__(self):
        return f"Conv2d({self.in_channels}, {self.out_channels}, kernel_size={self.kernel_size}, stride={self.stride}, padding={self.padding})"

In [16]:
conv = Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding='valid', bias=False)
x = np.arange(1, 10).reshape(1, 1, 3, 3).astype(float)

out = conv.forward(x)
print(out.shape)
print(out)

(1, 1, 1, 1)
[[[[0.86076257]]]]


In [17]:
gradOutput = np.ones_like(out)
gradInput = conv.updateGradInput(x, gradOutput)

print(gradInput.shape)
print(gradInput)

(1, 1, 3, 3)
[[[[ 0.12657164 -0.1157641  -0.28310137]
   [-0.0776917   0.10596869  0.09245947]
   [ 0.11698753 -0.2831753   0.27640883]]]]


#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [18]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=-np.inf
        )

        batch_size, channels, H_pad, W_pad = padded_input.shape

        out_H = (H_pad - kH) // sH + 1
        out_W = (W_pad - kW) // sW + 1

        self.output = np.zeros((batch_size, channels, out_H, out_W), dtype=input.dtype)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_H):
                    for j in range(out_W):
                        h_start = i * sH
                        h_end = h_start + kH
                        w_start = j * sW
                        w_end = w_start + kW

                        region = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.max(region)

        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=-np.inf
        )

        grad_padded = np.zeros_like(padded_input)

        batch_size, channels, H_pad, W_pad = padded_input.shape
        out_H, out_W = gradOutput.shape[2], gradOutput.shape[3]

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_H):
                    for j in range(out_W):
                        h_start = i * sH
                        h_end = h_start + kH
                        w_start = j * sW
                        w_end = w_start + kW

                        region = padded_input[n, c, h_start:h_end, w_start:w_end]
                        max_pos = np.unravel_index(np.argmax(region), region.shape)

                        grad_padded[n, c, h_start + max_pos[0], w_start + max_pos[1]] += gradOutput[n, c, i, j]

        if pH == 0 and pW == 0:
            self.gradInput = grad_padded
        else:
            self.gradInput = grad_padded[:, :, pH:H_pad - pH, pW:W_pad - pW]

        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"


class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=0
        )

        batch_size, channels, H_pad, W_pad = padded_input.shape

        out_H = (H_pad - kH) // sH + 1
        out_W = (W_pad - kW) // sW + 1

        self.output = np.zeros((batch_size, channels, out_H, out_W), dtype=input.dtype)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_H):
                    for j in range(out_W):
                        h_start = i * sH
                        h_end = h_start + kH
                        w_start = j * sW
                        w_end = w_start + kW

                        region = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.mean(region)

        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=0
        )

        grad_padded = np.zeros_like(padded_input)

        batch_size, channels, H_pad, W_pad = padded_input.shape
        out_H, out_W = gradOutput.shape[2], gradOutput.shape[3]

        coeff = 1.0 / (kH * kW)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_H):
                    for j in range(out_W):
                        h_start = i * sH
                        h_end = h_start + kH
                        w_start = j * sW
                        w_end = w_start + kW

                        grad_padded[n, c, h_start:h_end, w_start:w_end] += gradOutput[n, c, i, j] * coeff

        if pH == 0 and pW == 0:
            self.gradInput = grad_padded
        else:
            self.gradInput = grad_padded[:, :, pH:H_pad - pH, pW:W_pad - pW]

        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

In [19]:
pool = MaxPool2d(kernel_size=2, stride=2, padding=0)

x = np.array([[[[1., 2., 3., 4.],
                [5., 6., 7., 8.],
                [9.,10.,11.,12.],
                [13.,14.,15.,16.]]]])

out = pool.forward(x)
print(out.shape)
print(out)

(1, 1, 2, 2)
[[[[ 6.  8.]
   [14. 16.]]]]


In [20]:
gradOutput = np.ones_like(out)
gradInput = pool.updateGradInput(x, gradOutput)

print(gradInput.shape)
print(gradInput)

(1, 1, 4, 4)
[[[[0. 0. 0. 0.]
   [0. 1. 0. 1.]
   [0. 0. 0. 0.]
   [0. 1. 0. 1.]]]]


In [21]:
pool = AvgPool2d(kernel_size=2, stride=2, padding=0)

x = np.array([[[[1., 2., 3., 4.],
                [5., 6., 7., 8.],
                [9.,10.,11.,12.],
                [13.,14.,15.,16.]]]])

out = pool.forward(x)
print(out.shape)
print(out)

(1, 1, 2, 2)
[[[[ 3.5  5.5]
   [11.5 13.5]]]]


In [22]:
gradOutput = np.ones_like(out)
gradInput = pool.updateGradInput(x, gradOutput)

print(gradInput.shape)
print(gradInput)

(1, 1, 4, 4)
[[[[0.25 0.25 0.25 0.25]
   [0.25 0.25 0.25 0.25]
   [0.25 0.25 0.25 0.25]
   [0.25 0.25 0.25 0.25]]]]


#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [23]:
class GlobalMaxPool2d(Module):
    def updateOutput(self, input):
        self.input = input
        self.output = np.max(input, axis=(2, 3), keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.zeros_like(input)

        max_vals = np.max(input, axis=(2, 3), keepdims=True)
        mask = (input == max_vals)

        self.gradInput = mask * gradOutput

        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"

class GlobalAvgPool2d(Module):
    def updateOutput(self, input):
        self.input = input
        self.output = np.mean(input, axis=(2, 3), keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.zeros_like(input)

        _, _, H, W = input.shape
        coeff = 1.0 / (H * W)

        self.gradInput += gradOutput * coeff

        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"

In [24]:
x = np.array([[[[1., 2.],
                [3., 4.]]]])

pool = GlobalMaxPool2d()
print(pool.forward(x))

[[[[4.]]]]


#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [25]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.input_shape = input.shape
        ndim = len(input.shape)

        start = self.start_dim if self.start_dim >= 0 else ndim + self.start_dim
        end = self.end_dim if self.end_dim >= 0 else ndim + self.end_dim

        new_shape = (
            list(input.shape[:start]) +
            [int(np.prod(input.shape[start:end + 1]))] +
            list(input.shape[end + 1:])
        )

        self.output = input.reshape(new_shape)
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput.reshape(self.input_shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"

In [26]:
x = np.array([[[[1,2],[3,4]]]])

flat = Flatten()
out = flat.forward(x)

print(out.shape)
print(out)

(4,)
[1 2 3 4]


# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [27]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [28]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = np.where(input > 0, input, self.slope * input)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        grad = np.where(input > 0, 1, self.slope)
        self.gradInput = gradOutput * grad
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

In [29]:
x = np.array([[-2., -1., 0., 3.]])
act = LeakyReLU()

out = act.forward(x)
print(out)

[[-0.06 -0.03  0.    3.  ]]


## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [30]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = np.where(
            input > 0,
            input,
            self.alpha * (np.exp(input) - 1)
        )
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        grad = np.where(
            input > 0,
            1,
            self.output + self.alpha
        )
        self.gradInput = gradOutput * grad
        return self.gradInput

    def __repr__(self):
        return "ELU"

In [31]:
x = np.array([[-1., 0., 2.]])
act = ELU()

out = act.forward(x)
print(out)

[[-0.63212056  0.          2.        ]]


## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [32]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = np.log(1 + np.exp(input))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        sigmoid = 1 / (1 + np.exp(-input))
        self.gradInput = gradOutput * sigmoid
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

In [33]:
x = np.array([[-1., 0., 2.]])
act = SoftPlus()

out = act.forward(x)
print(out)

[[0.31326169 0.69314718 2.12692801]]


#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [34]:
class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.input = input
        c = np.sqrt(2 / np.pi)
        self.t = c * (input + 0.044715 * input**3)
        self.output = 0.5 * input * (1 + np.tanh(self.t))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        c = np.sqrt(2 / np.pi)
        tanh_t = np.tanh(self.t)

        dt_dx = c * (1 + 3 * 0.044715 * input**2)
        grad_gelu = 0.5 * (1 + tanh_t) + 0.5 * input * (1 - tanh_t**2) * dt_dx

        self.gradInput = gradOutput * grad_gelu
        return self.gradInput

    def __repr__(self):
        return "Gelu"

In [35]:
x = np.array([[-1., 0., 2.]])
act = Gelu()

out = act.forward(x)
print(out)

[[-0.15880801  0.          1.95459769]]


In [36]:
gradOutput = np.ones_like(out)
gradInput = act.updateGradInput(x, gradOutput)
print(gradInput)

[[-0.08296408  0.5         1.08609926]]


# Criterions

Criterions are used to score the models answers.

In [37]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [38]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [39]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        # Your code goes here. ################################################
        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        # Your code goes here. ################################################
        self.gradInput = -(target / input_clamp) / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

In [40]:
crit = ClassNLLCriterionUnstable()

input = np.array([[0.1, 0.6, 0.3],
                  [0.8, 0.1, 0.1]])

target = np.array([[0, 1, 0],
                   [1, 0, 0]])

loss = crit.forward(input, target)
grad = crit.backward(input, target)

print(loss)
print(grad)

0.3669845875401002
[[-0.         -0.83333333 -0.        ]
 [-0.625      -0.         -0.        ]]


## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [41]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        # Your code goes here. ################################################
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        # Your code goes here. ################################################
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

In [42]:
crit = ClassNLLCriterion()

input = np.log(np.array([[0.1, 0.6, 0.3],
                         [0.8, 0.1, 0.1]]))

target = np.array([[0, 1, 0],
                   [1, 0, 0]])

loss = crit.forward(input, target)
grad = crit.backward(input, target)

print(loss)
print(grad)

0.3669845875401002
[[ 0.  -0.5  0. ]
 [-0.5  0.   0. ]]


1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.